In [1]:
from IPython.display import Image

- references
    - https://docs.vllm.ai/en/latest/features/automatic_prefix_caching/#introduction
        - https://github.com/vllm-project/vllm/blob/main/examples/offline_inference/automatic_prefix_caching.py
    - https://x.com/hamzaelshafie/status/2046593204481368265
- Automatic Prefix Caching, APC: 优化多轮对话推理性能 (ttft)
    - 默认情况下，vLLM 的 Automatic Prefix Caching (APC) 是全局且跨用户（跨请求）共享的。

- 完整块缓存原则 (Only caches full blocks): vLLM 管理 KV Cache 的最小单位是 Block（块）。只有被 Token 填满的 Block 才会被纳入全局缓存池。
- 最长前缀匹配 (Reuses the longest matching prefix): 缓存的复用必须从序列的头部开始，严格按照顺序逐块匹配。
- 链式哈希计算 (Hash Generation): 每一个 Block 的哈希标识不仅仅取决于该块内部的 Tokens，还严格依赖于其前驱块（Parent block）的哈希值。此外，还会引入可选的附加哈希因子（如 LoRA 模型权重标识、多模态特征等）。

In [2]:
Image(url='https://pbs.twimg.com/media/HGbuTeAXIAApBEW?format=jpg&name=4096x4096')

- multi-gpu settings
    - 在缺乏智能路由的分布式集群中，“前缀缓存”非但无法提升性能，反而会引发算力与显存的双重灾难。
    - 缓存感知路由 (Cache-Aware Routing) / 一致性哈希 (Consistent Hashing)：
        - vLLM Router 会在接入层直接拦截请求，对请求的 Prompt 前缀（如相同的系统提示词、相同的多轮历史）进行哈希运算。通过建立哈希环（Hash Ring）机制，强制确保携带相同前缀的请求永远被引流到固定的 Worker 上。这被称为“会话亲和性（Session Affinity）”，是破局的直接手段。

In [3]:
Image(url='https://pbs.twimg.com/media/HGb1idJXYAAKnvl?format=jpg&name=4096x4096')